In [1]:
from pathlib import Path
import pandas as pd
import re
import time
from tqdm import tqdm

# 原始筛选结果目录
OLD_OUTPUT_DIR = Path(r"D:\Pycharm\JupyterProject1\ai_gait_restart")

# 新的 factor extraction 目录
FACTOR_OUTPUT_DIR = Path(r"D:\Pycharm\JupyterProject1\ai_gait_factor_restart")
FACTOR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输入：前面已经筛出的 Tier 1 文献
TIER1_CSV = OLD_OUTPUT_DIR / "06_tier1_llm_candidates.csv"

tier1_df = pd.read_csv(TIER1_CSV)

print("Tier 1 rows:", len(tier1_df))
print(tier1_df["rule_screening_label_v2"].value_counts())
print(tier1_df.columns.tolist())

Tier 1 rows: 200
rule_screening_label_v2
clinical_ai_gait_feature    70
target_stroke_ai_gait       67
target_neuro_ai_gait        45
ai_gait_feature_general     18
Name: count, dtype: int64
['dedup_key', 'retrieval_source', 'query_name', 'pmid', 'title', 'abstract', 'journal', 'year', 'doi', 'authors', 'url', 'fields_of_study', 'title_norm', 'doi_norm', 'citation_count', 'rule_screening_label_v2', 'llm_screening_tier']


In [2]:
from typing import List, Literal
from pydantic import BaseModel, Field


class GaitFactorExtraction(BaseModel):
    measurable_gait_factors: List[str] = Field(
        description=(
            "Specific measurable gait factors explicitly mentioned in the title or abstract. "
            "These are data-derived parameters that can be computed or estimated from video, pose landmarks, sensors, motion capture, or force data. "
            "Examples: gait speed, walking speed, cadence, step length, stride length, step width, stance time, swing time, double support time, "
            "single support time, gait phase, initial contact, toe-off, toe clearance, foot progression angle, joint angles, knee flexion, "
            "hip extension, ankle dorsiflexion, joint trajectories, range of motion, ground reaction force, joint moment, joint power, "
            "center of pressure, plantar pressure, body sway. "
            "Use an empty list if no specific measurable gait factor is explicitly stated."
        )
    )

    observable_gait_signs: List[str] = Field(
        description=(
            "Clinically observable gait signs or abnormalities explicitly mentioned in the title or abstract. "
            "These are signs a clinician might describe from watching a gait video. "
            "Examples: foot drop, toe drag, reduced toe clearance, delayed heel strike, forefoot contact, flat foot contact, knee hyperextension, "
            "reduced knee flexion, stiff-knee gait, reduced hip extension, hip hiking, circumduction, vaulting, trunk lean, forward trunk inclination, "
            "lateral trunk lean, pelvic obliquity, reduced arm swing, arm swing asymmetry, shoulder height asymmetry, body sway, turning instability. "
            "Use an empty list if no observable sign is explicitly stated."
        )
    )

    body_regions: List[
        Literal[
            "whole_body",
            "foot_ankle",
            "knee",
            "hip_pelvis",
            "trunk",
            "upper_limb_shoulder",
            "center_of_mass_or_pressure",
            "not_reported"
        ]
    ] = Field(
        description=(
            "Body regions involved in the extracted gait factors or signs. "
            "Use not_reported only if no body region can be identified from the title or abstract."
        )
    )

    gait_phases_or_tasks: List[
        Literal[
            "whole_gait_cycle",
            "initial_contact",
            "stance_phase",
            "swing_phase",
            "toe_off",
            "turning",
            "sit_to_stand",
            "stairs",
            "obstacle_crossing",
            "walking_endurance",
            "not_reported"
        ]
    ] = Field(
        description=(
            "Gait phase or functional walking task explicitly related to the extracted factors. "
            "Examples: stance phase, swing phase, initial contact, toe-off, turning, stairs, obstacle crossing, sit-to-stand, walking endurance."
        )
    )

    measurable_factor_families: List[
        Literal[
            "temporal_rhythm",
            "spatial_stepping",
            "foot_contact_clearance",
            "joint_kinematics",
            "kinetics_pressure",
            "symmetry_asymmetry",
            "posture_alignment",
            "upper_limb_movement",
            "stability_sway",
            "functional_task_performance",
            "not_reported"
        ]
    ] = Field(
        description=(
            "Families of directly measurable or computable gait factors. "
            "Do not include clinical scales here. "
            "temporal_rhythm = cadence, step time, stride time, stance time, swing time, double support; "
            "spatial_stepping = step length, stride length, step width, base of support; "
            "foot_contact_clearance = heel strike, initial contact, toe-off, toe clearance, foot progression angle; "
            "joint_kinematics = joint angles, joint trajectories, range of motion, knee/hip/ankle motion; "
            "kinetics_pressure = ground reaction force, joint moment, joint power, center of pressure, plantar pressure; "
            "symmetry_asymmetry = left-right asymmetry in time, distance, posture, or joint motion; "
            "posture_alignment = trunk, pelvis, shoulder, body alignment; "
            "upper_limb_movement = arm swing or shoulder movement; "
            "stability_sway = body sway, postural stability, turning stability; "
            "functional_task_performance = TUG components, stairs, turning, obstacle crossing, sit-to-stand, endurance tasks."
        )
    )

    observable_sign_families: List[
        Literal[
            "foot_clearance_impairment",
            "abnormal_foot_contact",
            "reduced_joint_motion",
            "excessive_joint_motion",
            "limb_stiffness",
            "gait_asymmetry",
            "compensatory_movement",
            "postural_abnormality",
            "balance_instability",
            "upper_limb_asymmetry",
            "not_reported"
        ]
    ] = Field(
        description=(
            "Families of clinically observable gait signs. "
            "These describe what a clinician might observe in a video. "
            "Do not use this field for generic measurable parameters unless they are described as abnormal signs."
        )
    )

    clinical_constructs: List[
        Literal[
            "walking_speed_capacity",
            "walking_endurance",
            "functional_mobility",
            "dynamic_balance",
            "gait_quality",
            "motor_control",
            "disease_severity",
            "fall_risk",
            "rehabilitation_outcome",
            "not_reported"
        ]
    ] = Field(
        description=(
            "Higher-level clinical constructs targeted by the study. "
            "These are not raw gait features. "
            "Examples: walking speed capacity, endurance, functional mobility, dynamic balance, gait quality, disease severity, fall risk."
        )
    )

    clinical_scale_or_outcome_targets: List[str] = Field(
        description=(
            "Clinical scales, clinical scores, functional tests, or outcomes explicitly mentioned. "
            "Examples: TUG, 10MWT, 6MWT, BBS, FGA, DGI, UPDRS, SARA, GDI, WGS, fall risk, disease severity, rehabilitation outcome. "
            "Use an empty list if no clinical scale or outcome target is explicitly stated."
        )
    )

    input_signal_used_for_factor: List[str] = Field(
        description=(
            "Input signals used to derive the gait factors. "
            "Examples: RGB video, monocular video, smartphone video, pose landmarks, keypoints, 3D joint trajectories, IMU, wearable sensors, "
            "pressure insole, force plate, optical motion capture, markerless motion capture, EMG."
        )
    )

    video_observable_level: Literal[
        "directly_video_observable",
        "pose_or_keypoint_computable",
        "sensor_or_force_only",
        "clinical_scale_only",
        "mixed",
        "not_reported"
    ] = Field(
        description=(
            "Whether the extracted factors are directly observable from video, computable from pose/keypoints, available only from sensors/force platforms, "
            "or only clinical scale targets."
        )
    )

    clinical_interpretability_level: Literal[
        "raw_parameter_only",
        "clinical_feature_description",
        "clinical_construct_mapping",
        "clinical_scale_or_outcome_prediction",
        "clinical_report_generation",
        "not_reported"
    ] = Field(
        description=(
            "How far the paper goes from raw gait factors toward clinical meaning. "
            "raw_parameter_only = only computes parameters; "
            "clinical_feature_description = describes abnormal gait signs or clinically meaningful features; "
            "clinical_construct_mapping = maps factors to constructs such as balance, mobility, gait quality, fall risk; "
            "clinical_scale_or_outcome_prediction = predicts clinical scale, score, disease severity, or outcome; "
            "clinical_report_generation = generates structured or narrative clinical gait report."
        )
    )

    factor_extraction_note: str = Field(
        description=(
            "Brief explanation of what the paper analyzes. Clearly distinguish measurable gait factors, observable signs, and clinical targets."
        )
    )

    extraction_confidence: Literal["high", "medium", "low"] = Field(
        description=(
            "high = title and abstract clearly state the factors; "
            "medium = factors are stated but broad or incomplete; "
            "low = abstract is missing, vague, or ambiguous."
        )
    )

In [4]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-5.5"


def extract_gait_factors_one_paper(title: str, abstract: str) -> GaitFactorExtraction:
    title = "" if pd.isna(title) else str(title)
    abstract = "" if pd.isna(abstract) else str(abstract)

    prompt = f"""
You are extracting fine-grained gait factors for a medical mapping review.

Research question:
What specific gait-related factors do AI-based gait analysis studies analyze?

Important:
Do NOT center the extraction around any single clinical scale.
Do NOT mix raw gait factors, clinical constructs, and clinical scales.
Your job is to separate:
1. measurable gait factors,
2. observable gait signs,
3. body regions,
4. gait phases or functional tasks,
5. clinical constructs,
6. clinical scales or outcome targets.

Strict rules:
1. Use only information explicitly stated in the title and abstract.
2. Do not decide whether the paper should be included or excluded.
3. Do not infer unstated factors, body regions, phases, clinical constructs, or scales.
4. If a factor is not explicitly stated, leave the corresponding list empty or use not_reported where required.
5. Be conservative. This is for medical research.
6. Do not treat generic phrases such as "gait analysis" or "mobility assessment" as specific gait factors.
7. Clinical scales or outcomes must not be counted as raw gait factors.
8. Balance, functional mobility, disease severity, fall risk, and rehabilitation outcome are clinical constructs or targets, not raw gait parameters.
9. Examples:
   - gait speed, cadence, stance time = measurable_gait_factors, temporal_rhythm.
   - step length, stride length, step width = measurable_gait_factors, spatial_stepping.
   - toe clearance, heel strike, toe-off = measurable_gait_factors, foot_contact_clearance.
   - knee flexion, hip extension, ankle dorsiflexion, joint angles = measurable_gait_factors, joint_kinematics.
   - ground reaction force, center of pressure = measurable_gait_factors, kinetics_pressure.
   - trunk lean, pelvic tilt, shoulder alignment = posture_alignment and possibly observable_gait_signs if described as abnormal.
   - foot drop, toe drag, circumduction, hip hiking, vaulting, knee hyperextension = observable_gait_signs.
   - TUG, 10MWT, 6MWT, BBS, FGA, DGI, UPDRS, SARA, GDI, WGS = clinical_scale_or_outcome_targets.
   - fall risk, disease severity, rehabilitation outcome = clinical constructs or outcome targets.

Title:
{title}

Abstract:
{abstract}
"""

    completion = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful biomedical literature extraction assistant. "
                    "Extract only explicitly stated gait-related factors. "
                    "Do not fabricate. Keep raw gait factors, observable signs, clinical constructs, and clinical scales separate."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format=GaitFactorExtraction,
    )

    return completion.choices[0].message.parsed

In [5]:
test_df = (
    tier1_df
    .groupby("rule_screening_label_v2", group_keys=False)
    .head(1)
    .copy()
)

print("Test rows:", len(test_df))
print(test_df[["title", "rule_screening_label_v2"]])

Test rows: 4
                                               title   rule_screening_label_v2
0  Closed-Loop Wearable Device Network of Intrins...     target_stroke_ai_gait
1  Sarcopenia classification model for musculoske...  clinical_ai_gait_feature
3  Gait Video-Based Prediction of Severity of Cer...      target_neuro_ai_gait
8  Predicting Knee Joint Contact Force Peaks Duri...   ai_gait_feature_general


In [6]:
test_results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        extracted = extract_gait_factors_one_paper(
            title=row.get("title", ""),
            abstract=row.get("abstract", "")
        )

        extracted_dict = extracted.model_dump()
        merged = row.to_dict()

        for key, value in extracted_dict.items():
            if isinstance(value, list):
                merged[f"factor_{key}"] = "; ".join(value)
            else:
                merged[f"factor_{key}"] = value

        merged["factor_llm_error"] = ""
        test_results.append(merged)

    except Exception as e:
        print("Error at row:", idx)
        print("Title:", row.get("title", ""))
        print("Error:", e)

        merged = row.to_dict()
        merged["factor_llm_error"] = str(e)
        test_results.append(merged)

    time.sleep(0.5)

test_factor_df = pd.DataFrame(test_results)

preview_cols = [
    "title",
    "rule_screening_label_v2",
    "factor_measurable_gait_factors",
    "factor_observable_gait_signs",
    "factor_body_regions",
    "factor_gait_phases_or_tasks",
    "factor_measurable_factor_families",
    "factor_observable_sign_families",
    "factor_clinical_constructs",
    "factor_clinical_scale_or_outcome_targets",
    "factor_input_signal_used_for_factor",
    "factor_video_observable_level",
    "factor_clinical_interpretability_level",
    "factor_factor_extraction_note",
    "factor_extraction_confidence",
    "factor_llm_error"
]

preview_cols = [c for c in preview_cols if c in test_factor_df.columns]

test_factor_df[preview_cols]

100%|██████████| 4/4 [01:31<00:00, 22.99s/it]


,title,rule_screening_label_v2,factor_measurable_gait_factors,factor_observable_gait_signs,factor_body_regions,factor_gait_phases_or_tasks,factor_measurable_factor_families,factor_observable_sign_families,factor_clinical_constructs,factor_clinical_scale_or_outcome_targets,factor_input_signal_used_for_factor,factor_video_observable_level,factor_clinical_interpretability_level,factor_factor_extraction_note,factor_extraction_confidence,factor_llm_error
0,Closed-Loop Wearable Device Network of Intrins...,target_stroke_ai_gait,ankle joint angle mean; ankle joint angle vari...,foot drop,foot_ankle,whole_gait_cycle,joint_kinematics,foot_clearance_impairment,functional_mobility; rehabilitation_outcome,oxyhemoglobin concentration tested by function...,surface myoelectricity; myoelectricity signal;...,mixed,clinical_scale_or_outcome_prediction,The abstract explicitly reports stroke patient...,high,
1,Sarcopenia classification model for musculoske...,clinical_ai_gait_feature,Hip_dif; Ankle_dif; Hipankle_dif,,foot_ankle; hip_pelvis,not_reported,joint_kinematics,not_reported,disease_severity,sarcopenia classification; control and sarcope...,smart insole; inertial measurement unit sensor...,mixed,clinical_scale_or_outcome_prediction,The study uses smart insole measurements and A...,medium,
2,Gait Video-Based Prediction of Severity of Cer...,target_neuro_ai_gait,body sway,body sway,whole_body,turning,stability_sway; functional_task_performance,balance_instability,disease_severity,timed up-and-go test; Scale for the Assessment...,two-dimensional videos; gait videos; key point...,pose_or_keypoint_computable,clinical_scale_or_outcome_prediction,The study applies pose estimation to 2D gait v...,high,
3,Predicting Knee Joint Contact Force Peaks Duri...,ai_gait_feature_general,knee joint contact force peaks; knee joint loa...,,whole_body; knee,whole_gait_cycle,temporal_rhythm; joint_kinematics; kinetics_pr...,not_reported,not_reported,,motion capture data; video camera data; inerti...,mixed,raw_parameter_only,The study predicts knee joint loading/contact ...,high,


In [7]:
TEST_FACTOR_XLSX = FACTOR_OUTPUT_DIR / "01_factor_extraction_test.xlsx"

test_factor_df.to_excel(TEST_FACTOR_XLSX, index=False)

print("Saved test factor extraction to:", TEST_FACTOR_XLSX)

Saved test factor extraction to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\01_factor_extraction_test.xlsx


In [8]:
OUTPUT_FACTOR_RAW_CSV = FACTOR_OUTPUT_DIR / "02_tier1_factor_extraction_raw.csv"
OUTPUT_FACTOR_RAW_XLSX = FACTOR_OUTPUT_DIR / "02_tier1_factor_extraction_raw.xlsx"

SAVE_EVERY = 5

# 断点续跑
if OUTPUT_FACTOR_RAW_CSV.exists():
    old_factor_df = pd.read_csv(OUTPUT_FACTOR_RAW_CSV)

    if "dedup_key" in old_factor_df.columns:
        done_keys = set(old_factor_df["dedup_key"].astype(str))
        key_col = "dedup_key"
    elif "title_norm" in old_factor_df.columns:
        done_keys = set(old_factor_df["title_norm"].astype(str))
        key_col = "title_norm"
    else:
        done_keys = set(old_factor_df["title"].astype(str))
        key_col = "title"

    print("Existing factor output found:", len(old_factor_df), "rows")
    print("Resume key:", key_col)

else:
    old_factor_df = pd.DataFrame()

    if "dedup_key" in tier1_df.columns:
        key_col = "dedup_key"
    elif "title_norm" in tier1_df.columns:
        key_col = "title_norm"
    else:
        key_col = "title"

    done_keys = set()
    print("No existing factor output found. Starting from zero.")
    print("Resume key:", key_col)


def save_factor_progress(old_df, new_results, output_csv, output_xlsx):
    temp_df = pd.concat([old_df, pd.DataFrame(new_results)], ignore_index=True)
    temp_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    temp_df.to_excel(output_xlsx, index=False)
    return temp_df


factor_results = []

for idx, row in tqdm(tier1_df.iterrows(), total=len(tier1_df)):
    key = str(row.get(key_col, row.get("title", "")))

    if key in done_keys:
        continue

    title = row.get("title", "")
    abstract = row.get("abstract", "")

    try:
        extracted = extract_gait_factors_one_paper(
            title=title,
            abstract=abstract
        )

        extracted_dict = extracted.model_dump()
        merged = row.to_dict()

        for field_name, value in extracted_dict.items():
            if isinstance(value, list):
                merged[f"factor_{field_name}"] = "; ".join(value)
            else:
                merged[f"factor_{field_name}"] = value

        merged["factor_llm_error"] = ""
        factor_results.append(merged)

    except Exception as e:
        print("\nError at row:", idx)
        print("Title:", title)
        print("Error:", e)

        merged = row.to_dict()
        merged["factor_llm_error"] = str(e)
        factor_results.append(merged)

    if len(factor_results) > 0 and len(factor_results) % SAVE_EVERY == 0:
        temp_df = save_factor_progress(
            old_factor_df,
            factor_results,
            OUTPUT_FACTOR_RAW_CSV,
            OUTPUT_FACTOR_RAW_XLSX
        )
        print("Progress saved:", len(temp_df), "rows")

    time.sleep(0.5)


final_factor_df = save_factor_progress(
    old_factor_df,
    factor_results,
    OUTPUT_FACTOR_RAW_CSV,
    OUTPUT_FACTOR_RAW_XLSX
)

print("\nDone.")
print("Saved CSV to:", OUTPUT_FACTOR_RAW_CSV)
print("Saved Excel to:", OUTPUT_FACTOR_RAW_XLSX)
print("Total output rows:", len(final_factor_df))

No existing factor output found. Starting from zero.
Resume key: dedup_key


  2%|▏         | 4/200 [01:37<1:11:34, 21.91s/it]

Progress saved: 5 rows


  4%|▍         | 9/200 [03:17<1:00:23, 18.97s/it]

Progress saved: 10 rows


  7%|▋         | 14/200 [05:05<1:04:50, 20.92s/it]

... (73 outputs left out)

100%|█████████▉| 199/200 [1:13:17<00:17, 17.75s/it]

Progress saved: 200 rows


100%|██████████| 200/200 [1:13:33<00:00, 22.07s/it]


Done.
Saved CSV to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\02_tier1_factor_extraction_raw.csv
Saved Excel to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\02_tier1_factor_extraction_raw.xlsx
Total output rows: 200


In [9]:
factor_df = pd.read_csv(OUTPUT_FACTOR_RAW_CSV)

print("Rows:", len(factor_df))
print("Errors:", factor_df["factor_llm_error"].astype(str).replace("nan", "").ne("").sum())

summary_cols = [
    "factor_video_observable_level",
    "factor_clinical_interpretability_level",
    "factor_extraction_confidence"
]

for col in summary_cols:
    print("\n====================")
    print(col)
    print("====================")
    print(factor_df[col].value_counts(dropna=False))

Rows: 200
Errors: 200

factor_video_observable_level
factor_video_observable_level
sensor_or_force_only           69
pose_or_keypoint_computable    61
mixed                          59
not_reported                    7
clinical_scale_only             2
directly_video_observable       2
Name: count, dtype: int64

factor_clinical_interpretability_level
factor_clinical_interpretability_level
raw_parameter_only                      83
clinical_scale_or_outcome_prediction    69
clinical_construct_mapping              35
clinical_feature_description            11
not_reported                             2
Name: count, dtype: int64

factor_extraction_confidence
factor_extraction_confidence
high      125
medium     71
low         4
Name: count, dtype: int64


In [10]:
def split_semicolon_items(x):
    if pd.isna(x):
        return []

    text = str(x).strip()
    if text == "" or text.lower() in ["nan", "none", "not_reported", "not reported"]:
        return []

    items = re.split(r";|,", text)
    return [
        item.strip()
        for item in items
        if item.strip() and item.strip().lower() not in ["nan", "none", "not_reported", "not reported"]
    ]


def value_counts_semicolon(df, col):
    temp = df[[col]].copy()
    temp[col] = temp[col].apply(split_semicolon_items)
    temp = temp.explode(col)
    temp = temp.dropna(subset=[col])
    temp = temp[temp[col].astype(str).str.strip() != ""]
    return temp[col].value_counts().reset_index().rename(
        columns={"index": col, col: "count"}
    )


measurable_factor_counts = value_counts_semicolon(
    factor_df,
    "factor_measurable_gait_factors"
)

observable_sign_counts = value_counts_semicolon(
    factor_df,
    "factor_observable_gait_signs"
)

body_region_counts = value_counts_semicolon(
    factor_df,
    "factor_body_regions"
)

phase_task_counts = value_counts_semicolon(
    factor_df,
    "factor_gait_phases_or_tasks"
)

measurable_family_counts = value_counts_semicolon(
    factor_df,
    "factor_measurable_factor_families"
)

observable_family_counts = value_counts_semicolon(
    factor_df,
    "factor_observable_sign_families"
)

clinical_construct_counts = value_counts_semicolon(
    factor_df,
    "factor_clinical_constructs"
)

clinical_scale_counts = value_counts_semicolon(
    factor_df,
    "factor_clinical_scale_or_outcome_targets"
)

print("Top measurable gait factors:")
display(measurable_factor_counts.head(30))

print("Top observable gait signs:")
display(observable_sign_counts.head(30))

print("Body regions:")
display(body_region_counts)

print("Measurable factor families:")
display(measurable_family_counts)

print("Observable sign families:")
display(observable_family_counts)

Top measurable gait factors:


,count,count
0,step length,19
1,walking speed,16
2,gait speed,15
3,joint angles,14
4,stride length,12
5,cadence,11
6,swing time,9
7,gait events,9
8,spatiotemporal gait parameters,9
9,stance time,8


Top observable gait signs:


,count,count
0,freezing of gait,3
1,foot drop,2
2,crouch gait,2
3,circumduction gait,2
4,equinus gait,2
5,ataxic gait,1
6,Stiff-Knee gait,1
7,"larger knee varus ""bump"" in swing",1
8,freezing episodes,1
9,walking difficulties,1


Body regions:


,count,count
0,whole_body,110
1,foot_ankle,105
2,knee,78
3,hip_pelvis,76
4,trunk,17
5,upper_limb_shoulder,9
6,center_of_mass_or_pressure,5


Measurable factor families:


,count,count
0,joint_kinematics,105
1,temporal_rhythm,98
2,spatial_stepping,69
3,kinetics_pressure,24
4,foot_contact_clearance,22
5,symmetry_asymmetry,17
6,stability_sway,11
7,functional_task_performance,10
8,posture_alignment,10
9,upper_limb_movement,4


Observable sign families:


,count,count
0,balance_instability,8
1,compensatory_movement,6
2,gait_asymmetry,4
3,foot_clearance_impairment,3
4,abnormal_foot_contact,3
5,limb_stiffness,2
6,excessive_joint_motion,2
7,postural_abnormality,2
8,reduced_joint_motion,2


In [11]:
FACTOR_SUMMARY_XLSX = FACTOR_OUTPUT_DIR / "03_factor_summary_tables.xlsx"

summary_tables = {
    "measurable_gait_factors": measurable_factor_counts,
    "observable_gait_signs": observable_sign_counts,
    "body_regions": body_region_counts,
    "gait_phases_or_tasks": phase_task_counts,
    "measurable_factor_families": measurable_family_counts,
    "observable_sign_families": observable_family_counts,
    "clinical_constructs": clinical_construct_counts,
    "clinical_scale_targets": clinical_scale_counts,
    "video_observable_level": factor_df["factor_video_observable_level"].value_counts().reset_index(),
    "clinical_interpretability": factor_df["factor_clinical_interpretability_level"].value_counts().reset_index(),
}

with pd.ExcelWriter(FACTOR_SUMMARY_XLSX, engine="openpyxl") as writer:
    for sheet_name, table in summary_tables.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Saved factor summary to:", FACTOR_SUMMARY_XLSX)

Saved factor summary to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\03_factor_summary_tables.xlsx


In [1]:
from pathlib import Path
import pandas as pd
import re

FACTOR_OUTPUT_DIR = Path(r"D:\Pycharm\JupyterProject1\ai_gait_factor_restart")

RAW_FACTOR_CSV = FACTOR_OUTPUT_DIR / "02_tier1_factor_extraction_raw.csv"

factor_df = pd.read_csv(RAW_FACTOR_CSV)

print("Rows:", len(factor_df))
print("Columns:", factor_df.columns.tolist())

if "factor_llm_error" in factor_df.columns:
    print("LLM errors:", factor_df["factor_llm_error"].fillna("").astype(str).ne("").sum())

Rows: 200
Columns: ['dedup_key', 'retrieval_source', 'query_name', 'pmid', 'title', 'abstract', 'journal', 'year', 'doi', 'authors', 'url', 'fields_of_study', 'title_norm', 'doi_norm', 'citation_count', 'rule_screening_label_v2', 'llm_screening_tier', 'factor_measurable_gait_factors', 'factor_observable_gait_signs', 'factor_body_regions', 'factor_gait_phases_or_tasks', 'factor_measurable_factor_families', 'factor_observable_sign_families', 'factor_clinical_constructs', 'factor_clinical_scale_or_outcome_targets', 'factor_input_signal_used_for_factor', 'factor_video_observable_level', 'factor_clinical_interpretability_level', 'factor_factor_extraction_note', 'factor_extraction_confidence', 'factor_llm_error']
LLM errors: 0


In [2]:
def split_items(x):
    """
    Split semicolon/comma-separated items from LLM output.
    """
    if pd.isna(x):
        return []

    text = str(x).strip()

    if text == "" or text.lower() in ["nan", "none", "not_reported", "not reported"]:
        return []

    items = re.split(r";|,", text)

    cleaned = []
    for item in items:
        item = item.strip()
        if item and item.lower() not in ["nan", "none", "not_reported", "not reported"]:
            cleaned.append(item)

    return cleaned


def join_items(items):
    """
    Join unique items while preserving order.
    """
    seen = []
    for item in items:
        item = str(item).strip()
        if item and item not in seen:
            seen.append(item)
    return "; ".join(seen)


def text_has_any(text, terms):
    text = str(text).lower()
    return any(term.lower() in text for term in terms)

In [3]:
def clean_clinical_targets(x):
    items = split_items(x)

    clinical_scales = []
    clinical_outcomes = []
    model_metrics = []
    nonclinical_or_groups = []

    for item in items:
        t = item.lower()

        # Clinical scales / functional tests
        if text_has_any(t, ["timed up", "tug"]):
            clinical_scales.append("TUG")
        elif text_has_any(t, ["10mwt", "10-meter", "10 metre", "10 m walk", "10-meter walk", "10 metre walk"]):
            clinical_scales.append("10MWT")
        elif text_has_any(t, ["6mwt", "6-minute", "6 minute", "6-min"]):
            clinical_scales.append("6MWT")
        elif text_has_any(t, ["berg balance", "bbs"]):
            clinical_scales.append("BBS")
        elif text_has_any(t, ["functional gait assessment", "fga"]):
            clinical_scales.append("FGA")
        elif text_has_any(t, ["dynamic gait index", "dgi"]):
            clinical_scales.append("DGI")
        elif text_has_any(t, ["updrs", "unified parkinson", "mds-updrs"]):
            clinical_scales.append("UPDRS")
        elif text_has_any(t, ["sara", "scale for the assessment and rating of ataxia"]):
            clinical_scales.append("SARA")
        elif text_has_any(t, ["gait deviation index", "gdi"]):
            clinical_scales.append("GDI")
        elif text_has_any(t, ["wisconsin gait scale", "wgs"]):
            clinical_scales.append("WGS")
        elif text_has_any(t, ["gross motor function classification", "gmfcs"]):
            clinical_scales.append("GMFCS")
        elif text_has_any(t, ["functional movement scale"]):
            clinical_scales.append("Functional Movement Scale")
        elif text_has_any(t, ["harris hip score"]):
            clinical_scales.append("Harris Hip Score")

        # Model performance metrics
        elif text_has_any(t, [
            "accuracy", "precision", "recall", "f1", "auc", "sensitivity",
            "specificity", "classification metrics", "classification metric",
            "mean absolute error", "mae", "rmse", "r2", "correlation",
            "error", "performance metric"
        ]):
            model_metrics.append(item)

        # Physiological / raw signals, not clinical scale
        elif text_has_any(t, [
            "oxyhemoglobin", "deoxyhemoglobin", "hemoglobin",
            "fnirs", "eeg", "emg amplitude", "muscle activation",
            "electromyography signal", "brain activity"
        ]):
            nonclinical_or_groups.append(item)

        # Experimental groups / population labels
        elif text_has_any(t, [
            "control group", "healthy group", "patient group",
            "sarcopenia group", "control and sarcopenia",
            "healthy controls", "disease group",
            "patient vs healthy", "healthy vs patient"
        ]):
            nonclinical_or_groups.append(item)

        # Clinical outcomes / diagnostic targets / severity
        elif text_has_any(t, [
            "fall risk", "falls", "faller", "disease severity", "severity",
            "rehabilitation outcome", "functional recovery",
            "functional independence", "disease progression",
            "diagnosis", "diagnostic", "classification",
            "sarcopenia", "parkinson", "stroke", "ataxia",
            "osteoarthritis", "impairment", "walking ability",
            "recovery tendency", "post-tka", "knee arthroplasty"
        ]):
            clinical_outcomes.append(item)

        # Ambiguous leftover: keep as outcome candidate, not scale
        else:
            clinical_outcomes.append(item)

    return pd.Series({
        "clinical_scale_targets_cleaned": join_items(clinical_scales) if clinical_scales else "not_reported",
        "clinical_outcome_targets_cleaned": join_items(clinical_outcomes) if clinical_outcomes else "not_reported",
        "model_performance_metrics_cleaned": join_items(model_metrics),
        "nonclinical_signals_or_groups_cleaned": join_items(nonclinical_or_groups)
    })


clinical_target_cleaned = factor_df["factor_clinical_scale_or_outcome_targets"].apply(clean_clinical_targets)

factor_df = pd.concat([factor_df, clinical_target_cleaned], axis=1)

factor_df[[
    "factor_clinical_scale_or_outcome_targets",
    "clinical_scale_targets_cleaned",
    "clinical_outcome_targets_cleaned",
    "model_performance_metrics_cleaned",
    "nonclinical_signals_or_groups_cleaned"
]].head(20)

,factor_clinical_scale_or_outcome_targets,clinical_scale_targets_cleaned,clinical_outcome_targets_cleaned,model_performance_metrics_cleaned,nonclinical_signals_or_groups_cleaned
0,walking ability; functional movement scales; p...,Functional Movement Scale,walking ability; recovery tendency,,patient vs healthy classification
1,gait speed test; chair stand test; control vs ...,not_reported,gait speed test; chair stand test,,control vs sarcopenia group classification
2,TKA outcomes; treatment efficacy of total knee...,not_reported,TKA outcomes; treatment efficacy of total knee...,,
3,Timed Up-and-Go test; Scale for the Assessment...,TUG; SARA,ataxia severity,,
4,MDS-UPDRS; MDS-UPDRS gait impairment severity;...,UPDRS,motor impairment severity,,
5,unstable knee diagnosis; severity of the ACL i...,not_reported,unstable knee diagnosis; severity of the ACL i...,,
6,quadriceps hyperreflexia; rectus femoris react...,not_reported,quadriceps hyperreflexia; rectus femoris react...,,
7,NaN,not_reported,not_reported,,
8,NaN,not_reported,not_reported,,
9,Appendicular Skeletal Muscle Mass Index; Grip ...,not_reported,Appendicular Skeletal Muscle Mass Index; Grip ...,,


In [4]:
def classify_measurable_factor_terms(x):
    items = split_items(x)
    text = " ; ".join(items).lower()

    groups = []

    if text_has_any(text, [
        "cadence", "step time", "stride time", "stance time", "swing time",
        "double support", "single support", "gait phase", "initial contact",
        "toe-off", "toe off", "heel strike", "walking speed", "gait speed",
        "gait events", "gait event"
    ]):
        groups.append("temporal_rhythm")

    if text_has_any(text, [
        "step length", "stride length", "step width", "base of support",
        "walking distance", "stride width"
    ]):
        groups.append("spatial_stepping")

    if text_has_any(text, [
        "toe clearance", "foot clearance", "foot progression", "heel strike",
        "initial contact", "toe-off", "toe off", "foot angle",
        "foot rollover", "foot placement"
    ]):
        groups.append("foot_contact_clearance")

    if text_has_any(text, [
        "joint angle", "joint angles", "knee flexion", "knee angle",
        "hip flexion", "hip extension", "hip angle",
        "ankle dorsiflexion", "ankle angle", "plantarflexion",
        "joint trajectory", "joint trajectories", "range of motion",
        "rom", "kinematic", "body segment angle",
        "ankle kinematics", "knee kinematics", "hip kinematics"
    ]):
        groups.append("joint_kinematics")

    if text_has_any(text, [
        "ground reaction force", "grf", "joint moment", "joint power",
        "contact force", "center of pressure", "centre of pressure",
        "plantar pressure", "pressure", "knee joint contact force"
    ]):
        groups.append("kinetics_pressure")

    if text_has_any(text, [
        "symmetry", "asymmetry", "left-right", "left right",
        "bilateral difference", "side difference"
    ]):
        groups.append("symmetry_asymmetry")

    if text_has_any(text, [
        "trunk", "pelvic", "pelvis", "shoulder", "posture",
        "alignment", "inclination", "lean", "tilt", "rotation",
        "spinal alignment"
    ]):
        groups.append("posture_alignment")

    if text_has_any(text, [
        "arm swing", "upper limb", "shoulder movement", "arm movement"
    ]):
        groups.append("upper_limb_movement")

    if text_has_any(text, [
        "body sway", "postural stability", "stability", "balance",
        "center of mass", "centre of mass", "com"
    ]):
        groups.append("stability_sway")

    if text_has_any(text, [
        "turning", "sit-to-stand", "sit to stand", "stairs",
        "obstacle", "timed up", "tug", "walking endurance",
        "functional task"
    ]):
        groups.append("functional_task_performance")

    if not groups:
        groups.append("not_reported_or_unclear")

    return join_items(groups)


factor_df["measurable_factor_families_cleaned"] = factor_df[
    "factor_measurable_gait_factors"
].apply(classify_measurable_factor_terms)

In [5]:
def classify_observable_sign_terms(x):
    items = split_items(x)
    text = " ; ".join(items).lower()

    groups = []

    if text_has_any(text, [
        "toe drag", "foot drop", "reduced toe clearance",
        "insufficient toe clearance", "poor foot clearance",
        "low toe clearance"
    ]):
        groups.append("foot_clearance_impairment")

    if text_has_any(text, [
        "forefoot contact", "flat foot", "delayed heel strike",
        "abnormal foot contact", "foot slap", "equinus gait"
    ]):
        groups.append("abnormal_foot_contact")

    if text_has_any(text, [
        "reduced knee flexion", "reduced hip extension",
        "limited ankle dorsiflexion", "reduced joint motion",
        "limited range of motion", "reduced flexion",
        "reduced hip flexion"
    ]):
        groups.append("reduced_joint_motion")

    if text_has_any(text, [
        "knee hyperextension", "excessive plantarflexion",
        "excessive dorsiflexion", "excessive extension"
    ]):
        groups.append("excessive_joint_motion")

    if text_has_any(text, [
        "stiff knee", "stiff-knee", "limb stiffness",
        "stiff swing", "stiff gait", "crouch gait"
    ]):
        groups.append("limb_stiffness")

    if text_has_any(text, [
        "asymmetry", "asymmetric", "left-right",
        "uneven", "side difference"
    ]):
        groups.append("gait_asymmetry")

    if text_has_any(text, [
        "circumduction", "hip hiking", "vaulting",
        "trunk compensation", "compensatory", "compensation"
    ]):
        groups.append("compensatory_movement")

    if text_has_any(text, [
        "trunk lean", "forward trunk", "lateral trunk",
        "pelvic obliquity", "postural abnormality",
        "abnormal posture", "inclination"
    ]):
        groups.append("postural_abnormality")

    if text_has_any(text, [
        "body sway", "instability", "turning instability",
        "near-fall", "fall risk", "poor balance",
        "walking imbalance", "gait unsteadiness",
        "freezing of gait", "freezing episodes", "freezing"
    ]):
        groups.append("balance_instability")

    if text_has_any(text, [
        "reduced arm swing", "arm swing asymmetry",
        "shoulder asymmetry", "shoulder height"
    ]):
        groups.append("upper_limb_asymmetry")

    if not groups:
        groups.append("not_reported_or_unclear")

    return join_items(groups)


factor_df["observable_sign_families_cleaned"] = factor_df[
    "factor_observable_gait_signs"
].apply(classify_observable_sign_terms)

In [6]:
def normalize_measurable_factor_item(item):
    t = str(item).lower().strip()

    mapping_rules = [
        (["walking speed", "gait speed", "speed"], "gait/walking speed"),
        (["cadence"], "cadence"),
        (["step length"], "step length"),
        (["stride length"], "stride length"),
        (["step width", "stride width", "base of support"], "step/stride width"),
        (["step time"], "step time"),
        (["stride time"], "stride time"),
        (["stance time"], "stance time"),
        (["swing time"], "swing time"),
        (["double support"], "double support time"),
        (["single support"], "single support time"),
        (["gait event", "initial contact", "heel strike", "toe-off", "toe off"], "gait events"),
        (["toe clearance", "foot clearance"], "toe/foot clearance"),
        (["foot progression"], "foot progression angle"),
        (["joint angle", "joint angles"], "joint angles"),
        (["joint trajectory", "joint trajectories"], "joint trajectories"),
        (["knee flexion", "knee angle", "knee kinematics"], "knee kinematics"),
        (["hip flexion", "hip extension", "hip angle", "hip kinematics"], "hip kinematics"),
        (["ankle dorsiflexion", "ankle angle", "ankle kinematics", "plantarflexion"], "ankle kinematics"),
        (["range of motion", "rom"], "range of motion"),
        (["ground reaction force", "grf"], "ground reaction force"),
        (["joint moment"], "joint moment"),
        (["joint power"], "joint power"),
        (["center of pressure", "centre of pressure"], "center of pressure"),
        (["plantar pressure", "pressure"], "plantar pressure"),
        (["body sway"], "body sway"),
        (["trunk", "posture", "alignment", "pelvic", "pelvis", "shoulder"], "posture/alignment"),
        (["arm swing", "upper limb"], "upper-limb movement"),
        (["symmetry", "asymmetry"], "symmetry/asymmetry"),
        (["turning"], "turning performance"),
        (["stairs"], "stair negotiation"),
        (["obstacle"], "obstacle crossing"),
    ]

    for keywords, normalized in mapping_rules:
        if any(k in t for k in keywords):
            return normalized

    return item.strip()


def normalize_measurable_factors_column(x):
    items = split_items(x)
    normalized = [normalize_measurable_factor_item(item) for item in items]
    return join_items(normalized) if normalized else "not_reported"


factor_df["measurable_gait_factors_cleaned"] = factor_df[
    "factor_measurable_gait_factors"
].apply(normalize_measurable_factors_column)

In [7]:
def normalize_observable_sign_item(item):
    t = str(item).lower().strip()

    mapping_rules = [
        (["foot drop"], "foot drop"),
        (["toe drag"], "toe drag"),
        (["reduced toe clearance", "poor foot clearance", "insufficient toe clearance"], "reduced toe/foot clearance"),
        (["delayed heel strike"], "delayed heel strike"),
        (["forefoot contact"], "forefoot contact"),
        (["flat foot"], "flat foot contact"),
        (["knee hyperextension"], "knee hyperextension"),
        (["reduced knee flexion"], "reduced knee flexion"),
        (["stiff-knee", "stiff knee"], "stiff-knee gait"),
        (["reduced hip flexion"], "reduced hip flexion"),
        (["reduced hip extension"], "reduced hip extension"),
        (["circumduction"], "circumduction gait"),
        (["hip hiking"], "hip hiking"),
        (["vaulting"], "vaulting"),
        (["trunk lean", "forward trunk", "lateral trunk"], "trunk lean"),
        (["trunk compensation"], "trunk compensation"),
        (["pelvic obliquity"], "pelvic obliquity"),
        (["reduced arm swing"], "reduced arm swing"),
        (["arm swing asymmetry"], "arm swing asymmetry"),
        (["shoulder asymmetry", "shoulder height"], "shoulder asymmetry"),
        (["body sway"], "body sway"),
        (["turning instability"], "turning instability"),
        (["gait unsteadiness", "walking imbalance"], "gait unsteadiness"),
        (["freezing of gait", "freezing episodes"], "freezing of gait"),
        (["crouch gait"], "crouch gait"),
        (["equinus gait"], "equinus gait"),
        (["ataxic gait"], "ataxic gait"),
        (["asymmetric gait"], "asymmetric gait"),
    ]

    for keywords, normalized in mapping_rules:
        if any(k in t for k in keywords):
            return normalized

    return item.strip()


def normalize_observable_signs_column(x):
    items = split_items(x)
    normalized = [normalize_observable_sign_item(item) for item in items]
    return join_items(normalized) if normalized else "not_reported"


factor_df["observable_gait_signs_cleaned"] = factor_df[
    "factor_observable_gait_signs"
].apply(normalize_observable_signs_column)

In [8]:
def clean_clinical_interpretability(row):
    raw_level = str(row.get("factor_clinical_interpretability_level", "")).lower()

    scales = str(row.get("clinical_scale_targets_cleaned", "")).lower()
    outcomes = str(row.get("clinical_outcome_targets_cleaned", "")).lower()
    obs_signs = str(row.get("observable_gait_signs_cleaned", "")).lower()
    note = str(row.get("factor_factor_extraction_note", "")).lower()

    has_scale = scales not in ["", "nan", "not_reported", "not reported"]
    has_outcome = outcomes not in ["", "nan", "not_reported", "not reported"]
    has_observable_sign = obs_signs not in ["", "nan", "not_reported", "not reported"]

    if text_has_any(note, ["clinical report", "gait report", "narrative report", "structured report"]):
        return "clinical_report_generation"

    if has_scale or has_outcome:
        if text_has_any(note, ["predict", "prediction", "classify", "classification", "estimate", "score", "severity"]):
            return "clinical_scale_or_outcome_prediction"
        else:
            return "clinical_construct_mapping"

    if has_observable_sign:
        return "clinical_feature_description"

    if raw_level in [
        "raw_parameter_only",
        "clinical_feature_description",
        "clinical_construct_mapping",
        "clinical_scale_or_outcome_prediction",
        "clinical_report_generation",
        "not_reported"
    ]:
        return raw_level

    return "raw_parameter_only"


factor_df["clinical_interpretability_level_cleaned"] = factor_df.apply(
    clean_clinical_interpretability,
    axis=1
)

In [9]:
def clean_video_observable_level(row):
    input_signal = str(row.get("factor_input_signal_used_for_factor", "")).lower()
    measurable = str(row.get("measurable_gait_factors_cleaned", "")).lower()
    observable = str(row.get("observable_gait_signs_cleaned", "")).lower()

    has_video = text_has_any(input_signal, [
        "video", "camera", "monocular", "rgb", "smartphone"
    ])

    has_pose = text_has_any(input_signal + " " + measurable, [
        "pose", "keypoint", "skeleton", "joint trajectory", "joint trajectories",
        "joint angle", "markerless"
    ])

    has_sensor_force = text_has_any(input_signal, [
        "imu", "wearable", "inertial", "force plate", "pressure",
        "emg", "motion capture", "mocap", "optical motion capture"
    ])

    has_observable = observable not in ["", "nan", "not_reported", "not reported"]

    if has_video and has_observable:
        return "directly_video_observable"

    if has_video or has_pose:
        return "pose_or_keypoint_computable"

    if has_sensor_force:
        return "sensor_or_force_only"

    if str(row.get("clinical_scale_targets_cleaned", "")).lower() not in ["", "nan", "not_reported"]:
        return "clinical_scale_only"

    return "not_reported"


factor_df["video_observable_level_cleaned"] = factor_df.apply(
    clean_video_observable_level,
    axis=1
)

In [10]:
CLEANED_FACTOR_CSV = FACTOR_OUTPUT_DIR / "04_tier1_factor_extraction_cleaned.csv"
CLEANED_FACTOR_XLSX = FACTOR_OUTPUT_DIR / "04_tier1_factor_extraction_cleaned.xlsx"

factor_df.to_csv(CLEANED_FACTOR_CSV, index=False, encoding="utf-8-sig")
factor_df.to_excel(CLEANED_FACTOR_XLSX, index=False)

print("Saved cleaned factor CSV to:", CLEANED_FACTOR_CSV)
print("Saved cleaned factor Excel to:", CLEANED_FACTOR_XLSX)

Saved cleaned factor CSV to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\04_tier1_factor_extraction_cleaned.csv
Saved cleaned factor Excel to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\04_tier1_factor_extraction_cleaned.xlsx


In [11]:
def count_semicolon_values(df, col):
    temp = df[[col]].copy()
    temp[col] = temp[col].apply(split_items)
    temp = temp.explode(col)
    temp = temp.dropna(subset=[col])
    temp = temp[
        (temp[col].astype(str).str.strip() != "") &
        (~temp[col].astype(str).str.lower().isin(["not_reported", "not reported", "nan"]))
    ]

    out = temp[col].value_counts().reset_index()
    out.columns = [col, "count"]
    return out


summary_tables = {
    "measurable_gait_factors": count_semicolon_values(factor_df, "measurable_gait_factors_cleaned"),
    "observable_gait_signs": count_semicolon_values(factor_df, "observable_gait_signs_cleaned"),
    "measurable_factor_families": count_semicolon_values(factor_df, "measurable_factor_families_cleaned"),
    "observable_sign_families": count_semicolon_values(factor_df, "observable_sign_families_cleaned"),
    "body_regions": count_semicolon_values(factor_df, "factor_body_regions"),
    "gait_phases_or_tasks": count_semicolon_values(factor_df, "factor_gait_phases_or_tasks"),
    "clinical_constructs": count_semicolon_values(factor_df, "factor_clinical_constructs"),
    "clinical_scale_targets": count_semicolon_values(factor_df, "clinical_scale_targets_cleaned"),
    "clinical_outcome_targets": count_semicolon_values(factor_df, "clinical_outcome_targets_cleaned"),
    "model_metrics_removed": count_semicolon_values(factor_df, "model_performance_metrics_cleaned"),
    "nonclinical_removed": count_semicolon_values(factor_df, "nonclinical_signals_or_groups_cleaned"),
}

# 单列 value_counts 单独处理，避免列名混乱
video_level = factor_df["video_observable_level_cleaned"].value_counts().reset_index()
video_level.columns = ["video_observable_level_cleaned", "count"]

interpret_level = factor_df["clinical_interpretability_level_cleaned"].value_counts().reset_index()
interpret_level.columns = ["clinical_interpretability_level_cleaned", "count"]

summary_tables["video_observable_level"] = video_level
summary_tables["clinical_interpretability"] = interpret_level


FACTOR_SUMMARY_CLEANED_XLSX = FACTOR_OUTPUT_DIR / "05_factor_summary_cleaned.xlsx"

with pd.ExcelWriter(FACTOR_SUMMARY_CLEANED_XLSX, engine="openpyxl") as writer:
    for sheet_name, table in summary_tables.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Saved cleaned factor summary to:", FACTOR_SUMMARY_CLEANED_XLSX)

Saved cleaned factor summary to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\05_factor_summary_cleaned.xlsx


In [12]:
print("Measurable gait factors:")
display(summary_tables["measurable_gait_factors"].head(30))

print("Observable gait signs:")
display(summary_tables["observable_gait_signs"].head(30))

print("Measurable factor families:")
display(summary_tables["measurable_factor_families"])

print("Observable sign families:")
display(summary_tables["observable_sign_families"])

print("Body regions:")
display(summary_tables["body_regions"])

print("Gait phases/tasks:")
display(summary_tables["gait_phases_or_tasks"])

print("Clinical scale targets:")
display(summary_tables["clinical_scale_targets"])

print("Clinical interpretability:")
display(summary_tables["clinical_interpretability"])

print("Video observable level:")
display(summary_tables["video_observable_level"])

Measurable gait factors:


,measurable_gait_factors_cleaned,count
0,joint angles,41
1,gait/walking speed,36
2,knee kinematics,29
3,hip kinematics,21
4,step length,19
5,ankle kinematics,16
6,posture/alignment,16
7,symmetry/asymmetry,14
8,gait events,14
9,stride length,13


Observable gait signs:


,observable_gait_signs_cleaned,count
0,freezing of gait,5
1,circumduction gait,3
2,foot drop,2
3,stiff-knee gait,2
4,crouch gait,2
5,equinus gait,2
6,gait unsteadiness,2
7,ataxic gait,1
8,"larger knee varus ""bump"" in swing",1
9,walking difficulties,1


Measurable factor families:


,measurable_factor_families_cleaned,count
0,joint_kinematics,90
1,temporal_rhythm,66
2,not_reported_or_unclear,54
3,spatial_stepping,33
4,posture_alignment,21
5,symmetry_asymmetry,14
6,kinetics_pressure,14
7,foot_contact_clearance,13
8,stability_sway,12
9,functional_task_performance,1


... (8 outputs left out)

Clinical interpretability:


,clinical_interpretability_level_cleaned,count
0,raw_parameter_only,75
1,clinical_scale_or_outcome_prediction,61
2,clinical_construct_mapping,50
3,clinical_feature_description,12
4,not_reported,2


Video observable level:


,video_observable_level_cleaned,count
0,pose_or_keypoint_computable,107
1,sensor_or_force_only,57
2,directly_video_observable,16
3,not_reported,15
4,clinical_scale_only,5


In [13]:
from pathlib import Path
import pandas as pd
import re

FACTOR_OUTPUT_DIR = Path(r"D:\Pycharm\JupyterProject1\ai_gait_factor_restart")

CLEANED_FACTOR_XLSX = FACTOR_OUTPUT_DIR / "04_tier1_factor_extraction_cleaned.xlsx"

df = pd.read_excel(CLEANED_FACTOR_XLSX)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 200
Columns: ['dedup_key', 'retrieval_source', 'query_name', 'pmid', 'title', 'abstract', 'journal', 'year', 'doi', 'authors', 'url', 'fields_of_study', 'title_norm', 'doi_norm', 'citation_count', 'rule_screening_label_v2', 'llm_screening_tier', 'factor_measurable_gait_factors', 'factor_observable_gait_signs', 'factor_body_regions', 'factor_gait_phases_or_tasks', 'factor_measurable_factor_families', 'factor_observable_sign_families', 'factor_clinical_constructs', 'factor_clinical_scale_or_outcome_targets', 'factor_input_signal_used_for_factor', 'factor_video_observable_level', 'factor_clinical_interpretability_level', 'factor_factor_extraction_note', 'factor_extraction_confidence', 'factor_llm_error', 'clinical_scale_targets_cleaned', 'clinical_outcome_targets_cleaned', 'model_performance_metrics_cleaned', 'nonclinical_signals_or_groups_cleaned', 'measurable_factor_families_cleaned', 'observable_sign_families_cleaned', 'measurable_gait_factors_cleaned', 'observable_gait_signs_cle

In [14]:
def split_items(x):
    if pd.isna(x):
        return []

    text = str(x).strip()

    if text == "" or text.lower() in ["nan", "none", "not_reported", "not reported"]:
        return []

    items = re.split(r";|,", text)

    return [
        item.strip()
        for item in items
        if item.strip() and item.strip().lower() not in [
            "nan",
            "none",
            "not_reported",
            "not reported"
        ]
    ]


def join_items(items):
    seen = []
    for item in items:
        item = str(item).strip()
        if item and item not in seen:
            seen.append(item)
    return "; ".join(seen)


def count_semicolon_values(df, col):
    temp = df[[col]].copy()
    temp[col] = temp[col].apply(split_items)
    temp = temp.explode(col)
    temp = temp.dropna(subset=[col])

    temp = temp[
        (temp[col].astype(str).str.strip() != "") &
        (~temp[col].astype(str).str.lower().isin([
            "not_reported",
            "not reported",
            "nan",
            "not_reported_or_unclear"
        ]))
    ]

    out = temp[col].value_counts().reset_index()
    out.columns = [col, "count"]
    return out

In [15]:
def merge_family_columns(cleaned_col, raw_col):
    cleaned_items = split_items(cleaned_col)
    raw_items = split_items(raw_col)

    cleaned_valid = [
        x for x in cleaned_items
        if x not in ["not_reported_or_unclear", "not_reported", "not reported"]
    ]

    raw_valid = [
        x for x in raw_items
        if x not in ["not_reported_or_unclear", "not_reported", "not reported"]
    ]

    merged = cleaned_valid.copy()

    for item in raw_valid:
        if item not in merged:
            merged.append(item)

    if not merged:
        return "not_reported_or_unclear"

    return join_items(merged)


df["measurable_factor_families_final"] = df.apply(
    lambda row: merge_family_columns(
        row.get("measurable_factor_families_cleaned", ""),
        row.get("factor_measurable_factor_families", "")
    ),
    axis=1
)

df["observable_sign_families_final"] = df.apply(
    lambda row: merge_family_columns(
        row.get("observable_sign_families_cleaned", ""),
        row.get("factor_observable_sign_families", "")
    ),
    axis=1
)


In [16]:
FINAL_FACTOR_TABLE_XLSX = FACTOR_OUTPUT_DIR / "06_tier1_factor_extraction_final.xlsx"
FINAL_FACTOR_TABLE_CSV = FACTOR_OUTPUT_DIR / "06_tier1_factor_extraction_final.csv"

df.to_excel(FINAL_FACTOR_TABLE_XLSX, index=False)
df.to_csv(FINAL_FACTOR_TABLE_CSV, index=False, encoding="utf-8-sig")

print("Saved final factor table to:", FINAL_FACTOR_TABLE_XLSX)

Saved final factor table to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\06_tier1_factor_extraction_final.xlsx


In [17]:
summary_final = {
    "measurable_gait_factors": count_semicolon_values(df, "measurable_gait_factors_cleaned"),
    "observable_gait_signs": count_semicolon_values(df, "observable_gait_signs_cleaned"),
    "measurable_factor_families": count_semicolon_values(df, "measurable_factor_families_final"),
    "observable_sign_families": count_semicolon_values(df, "observable_sign_families_final"),
    "body_regions": count_semicolon_values(df, "factor_body_regions"),
    "gait_phases_or_tasks": count_semicolon_values(df, "factor_gait_phases_or_tasks"),
    "clinical_constructs": count_semicolon_values(df, "factor_clinical_constructs"),
    "clinical_scale_targets": count_semicolon_values(df, "clinical_scale_targets_cleaned"),
    "clinical_outcome_targets": count_semicolon_values(df, "clinical_outcome_targets_cleaned"),
}

video_level = df["video_observable_level_cleaned"].value_counts().reset_index()
video_level.columns = ["video_observable_level_cleaned", "count"]

interpret_level = df["clinical_interpretability_level_cleaned"].value_counts().reset_index()
interpret_level.columns = ["clinical_interpretability_level_cleaned", "count"]

summary_final["video_observable_level"] = video_level
summary_final["clinical_interpretability"] = interpret_level

In [18]:
FINAL_SUMMARY_XLSX = FACTOR_OUTPUT_DIR / "06_factor_summary_final.xlsx"

with pd.ExcelWriter(FINAL_SUMMARY_XLSX, engine="openpyxl") as writer:
    for sheet_name, table in summary_final.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Saved final summary to:", FINAL_SUMMARY_XLSX)

Saved final summary to: D:\Pycharm\JupyterProject1\ai_gait_factor_restart\06_factor_summary_final.xlsx


In [19]:
print("Measurable gait factors:")
display(summary_final["measurable_gait_factors"].head(30))

print("Observable gait signs:")
display(summary_final["observable_gait_signs"].head(30))

print("Measurable factor families:")
display(summary_final["measurable_factor_families"])

print("Observable sign families:")
display(summary_final["observable_sign_families"])

print("Body regions:")
display(summary_final["body_regions"])

print("Gait phases or tasks:")
display(summary_final["gait_phases_or_tasks"])

print("Clinical scale targets:")
display(summary_final["clinical_scale_targets"])

print("Clinical interpretability:")
display(summary_final["clinical_interpretability"])

print("Video observable level:")
display(summary_final["video_observable_level"])

Measurable gait factors:


,measurable_gait_factors_cleaned,count
0,joint angles,41
1,gait/walking speed,36
2,knee kinematics,29
3,hip kinematics,21
4,step length,19
5,ankle kinematics,16
6,posture/alignment,16
7,symmetry/asymmetry,14
8,gait events,14
9,stride length,13


Observable gait signs:


,observable_gait_signs_cleaned,count
0,freezing of gait,5
1,circumduction gait,3
2,foot drop,2
3,stiff-knee gait,2
4,crouch gait,2
5,equinus gait,2
6,gait unsteadiness,2
7,ataxic gait,1
8,"larger knee varus ""bump"" in swing",1
9,walking difficulties,1


Measurable factor families:


,measurable_factor_families_final,count
0,joint_kinematics,107
1,temporal_rhythm,100
2,spatial_stepping,69
3,kinetics_pressure,24
4,posture_alignment,23
5,foot_contact_clearance,22
6,symmetry_asymmetry,17
7,stability_sway,17
8,functional_task_performance,10
9,upper_limb_movement,4


... (8 outputs left out)

Clinical interpretability:


,clinical_interpretability_level_cleaned,count
0,raw_parameter_only,75
1,clinical_scale_or_outcome_prediction,61
2,clinical_construct_mapping,50
3,clinical_feature_description,12
4,not_reported,2


Video observable level:


,video_observable_level_cleaned,count
0,pose_or_keypoint_computable,107
1,sensor_or_force_only,57
2,directly_video_observable,16
3,not_reported,15
4,clinical_scale_only,5
